In [8]:
from ogb.nodeproppred import PygNodePropPredDataset, Evaluator
import torch_geometric.transforms as T
import torch
import numpy as np
import os.path as osp
from torch_geometric.nn import GCNConv, SAGEConv
import torch.nn.functional as F
from deeprobust.graph.utils import *
import torch_sparse
from deeprobust.graph.global_attack import Random
import logging
import time
from utils import scale_feats

root_path = osp.expanduser('/home/songsh/GCL/datasets_pyg')

dataname = 'ogbn-arxiv'
dataset = PygNodePropPredDataset(root=osp.join(root_path, 'OGB'), name=dataname,
                                     transform=T.ToSparseTensor())
data = dataset[0]

# process features
feat = data.x 
data.x = scale_feats(feat)

data.adj_t = data.adj_t.to_symmetric()
device = torch.device('cuda')
evaluator = Evaluator(name='ogbn-arxiv')

data = data.to(device)
split_idx = dataset.get_idx_split()
train_idx = split_idx['train'].to(device)
val_idx = split_idx['valid'].to(device)
test_idx = split_idx['test'].to(device)
ptb_rate = 0.1

In [9]:
adj = torch_sparse.tensor.SparseTensor.to_torch_sparse_coo_tensor(data.adj_t)
adj = to_scipy(adj)
n_perturbations = int(ptb_rate * (adj.sum()//2))
# Setup Attack Model
model = Random()
start = time.time()
model.attack(adj, n_perturbations)
print(time.time() - start)
adj = model.modified_adj
# DICE only attack the testing set
# heuristic_attack(adj, n_perturbations, val_idx, test_idx)
perturbed_adj = torch_sparse.tensor.from_scipy(adj)
data.adj_t = perturbed_adj.to(device)

359.2927885055542


In [10]:
def get_logger(filename, verbosity=1, name=None):
    level_dict = {0: logging.DEBUG, 1: logging.INFO, 2: logging.WARNING}
    formatter = logging.Formatter(
        "[%(asctime)s][%(filename)s][line:%(lineno)d][%(levelname)s] %(message)s"
    )
    logger = logging.getLogger(name)
    logger.setLevel(level_dict[verbosity])

    fh = logging.FileHandler(filename, "w")
    fh.setFormatter(formatter)
    logger.addHandler(fh)

    sh = logging.StreamHandler()
    sh.setFormatter(formatter)
    logger.addHandler(sh)

    return logger

logger = get_logger('./log/random_attack_%s_%s.log' % ('ogb', str(ptb_rate)))

In [11]:
class GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_layers,
                 dropout):
        super(GCN, self).__init__()

        self.convs = torch.nn.ModuleList()
        self.convs.append(GCNConv(in_channels, hidden_channels, cached=True))
        self.bns = torch.nn.ModuleList()
        self.bns.append(torch.nn.BatchNorm1d(hidden_channels))
        for _ in range(num_layers - 2):
            self.convs.append(
                GCNConv(hidden_channels, hidden_channels, cached=True))
            self.bns.append(torch.nn.BatchNorm1d(hidden_channels))
        self.convs.append(GCNConv(hidden_channels, out_channels, cached=True))

        self.dropout = dropout

    def reset_parameters(self):
        for conv in self.convs:
            conv.reset_parameters()
        for bn in self.bns:
            bn.reset_parameters()

    def forward(self, x, adj_t):
        for i, conv in enumerate(self.convs[:-1]):
            x = conv(x, adj_t)
            x = self.bns[i](x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.convs[-1](x, adj_t)
        return x.log_softmax(dim=-1)
    
    
def train(model, data, train_idx, optimizer):
    model.train()

    optimizer.zero_grad()
    out = model(data.x, data.adj_t)[train_idx]
    loss = F.nll_loss(out, data.y.squeeze(1)[train_idx])
    loss.backward()
    optimizer.step()

    return loss.item()


@torch.no_grad()
def test(model, data, split_idx, evaluator):
    model.eval()

    out = model(data.x, data.adj_t)
    y_pred = out.argmax(dim=-1, keepdim=True)

    train_acc = evaluator.eval({
        'y_true': data.y[split_idx['train']],
        'y_pred': y_pred[split_idx['train']],
    })['acc']
    valid_acc = evaluator.eval({
        'y_true': data.y[split_idx['valid']],
        'y_pred': y_pred[split_idx['valid']],
    })['acc']
    test_acc = evaluator.eval({
        'y_true': data.y[split_idx['test']],
        'y_pred': y_pred[split_idx['test']],
    })['acc']

    return train_acc, valid_acc, test_acc

In [12]:
gcn = GCN(in_channels=data.num_features,
          hidden_channels=256,
          out_channels=dataset.num_classes,
          num_layers=3,
          dropout=0.5).to(device)
for run in range(1):
    gcn.reset_parameters()
    optimizer = torch.optim.Adam(gcn.parameters(), lr=0.01)
    for epoch in range(1, 501):
        loss = train(gcn, data, train_idx, optimizer)
        result = test(gcn, data, split_idx, evaluator)
        if (epoch % 10) == 0:
            train_acc, valid_acc, test_acc = result
            logger.info(f'Run: {run + 1:02d}, '
                  f'Epoch: {epoch:02d}, '
                  f'Loss: {loss:.4f}, '
                  f'Train: {100 * train_acc:.2f}%, '
                  f'Valid: {100 * valid_acc:.2f}% '
                  f'Test: {100 * test_acc:.2f}%')

INFO:root:Run: 01, Epoch: 10, Loss: 1.4897, Train: 61.36%, Valid: 62.41% Test: 61.86%
[2023-12-10 18:58:32,491][1851050102.py][line:14][INFO] Run: 01, Epoch: 10, Loss: 1.4897, Train: 61.36%, Valid: 62.41% Test: 61.86%
[2023-12-10 18:58:32,491][1851050102.py][line:14][INFO] Run: 01, Epoch: 10, Loss: 1.4897, Train: 61.36%, Valid: 62.41% Test: 61.86%
INFO:root:Run: 01, Epoch: 20, Loss: 1.3167, Train: 64.85%, Valid: 65.43% Test: 65.10%
[2023-12-10 18:58:33,713][1851050102.py][line:14][INFO] Run: 01, Epoch: 20, Loss: 1.3167, Train: 64.85%, Valid: 65.43% Test: 65.10%
[2023-12-10 18:58:33,713][1851050102.py][line:14][INFO] Run: 01, Epoch: 20, Loss: 1.3167, Train: 64.85%, Valid: 65.43% Test: 65.10%
INFO:root:Run: 01, Epoch: 30, Loss: 1.2359, Train: 66.38%, Valid: 66.79% Test: 66.46%
[2023-12-10 18:58:34,935][1851050102.py][line:14][INFO] Run: 01, Epoch: 30, Loss: 1.2359, Train: 66.38%, Valid: 66.79% Test: 66.46%
[2023-12-10 18:58:34,935][1851050102.py][line:14][INFO] Run: 01, Epoch: 30, Loss: 

In [13]:
#save
import scipy
from datasets_dgl.utils import sparse_mx_to_torch_sparse_tensor
save_path = '/home/songsh/SPMGAE/datasets_dgl/all_data_attack/random/'

perturbed_adj = sparse_mx_to_torch_sparse_tensor(adj)
features  = data.x.cpu()
labels    = data.y.squeeze().cpu() # 需要squeeze到一维
train_idx = train_idx.cpu()
val_idx   = val_idx.cpu()
test_idx  = test_idx.cpu()


torch.save(perturbed_adj,   save_path +  '%s/random_%s_%s.pt' % (dataname,  dataname, ptb_rate))

if ptb_rate == 0: #保存一次label和features即可
    scipy.sparse.save_npz(save_path + '%s/%s_features' % (dataname,  dataname), scipy.sparse.csr_matrix(np.array(features)))
    np.save(save_path + '%s/%s_labels' % (dataname, dataname),   labels)

np.save( save_path + '%s/random_%s_%s_idx_train' % (dataname, dataname, ptb_rate),   train_idx)
np.save( save_path + '%s/random_%s_%s_idx_val'   % (dataname, dataname, ptb_rate),     val_idx)
np.save( save_path + '%s/random_%s_%s_idx_test'   % (dataname, dataname, ptb_rate),    test_idx)

In [14]:
from   datasets_dgl.data_dgl  import load_attack_data
from     easydict        import EasyDict

# dataset_name = dataset_name.split('-')[1]
DATASET = EasyDict()
DATASET.ATTACK = {
    "data"    : 'Attack-ogbn-arxiv',
    "attack"  : 'random',
    "ptb_rate": str(ptb_rate)
}
# now just attack use
dataset  = load_attack_data(DATASET['ATTACK'])
graph = dataset.graph
graph

[I] Loading dataset Attack-ogbn-arxiv...
train_mask, test_mask, val_mask sizes : tensor(90941) tensor(29799) tensor(48603)
[I] Finished loading.
[I] Data load time: 4.6382s


Graph(num_nodes=169343, num_edges=2716499,
      ndata_schemes={'feat': Scheme(shape=(128,), dtype=torch.float32), 'label': Scheme(shape=(), dtype=torch.int64), 'train_mask': Scheme(shape=(), dtype=torch.bool), 'val_mask': Scheme(shape=(), dtype=torch.bool), 'test_mask': Scheme(shape=(), dtype=torch.bool)}
      edata_schemes={})